# Introduction to Plotly

*© 2026 Dr. Dina Deifallah*

---
## What is Plotly?

- A **JavaScript visualisation library** with a Python wrapper
- Allows us to create interactive charts directly from Python
- Charts can be embedded in dashboards and web applications using [Streamlit](https://streamlit.io/), [Dash](https://dash.plotly.com/), or [Chart Studio](https://chart-studio.plotly.com/)

## Why Plotly?

| Advantage | Detail |
|---|---|
| ⚡ Fast to use | Simple charts in a single line with Plotly Express |
| 🎨 Highly customisable | Fine-grained control via Graph Objects |
| 🖱️ Interactive by default | Hover tooltips, zoom, pan — no extra code needed |
| 📦 Dashboard-ready | First-class integration with Streamlit (Part II of this course) |

---
## 1. Structure of a Plotly Figure

Every Plotly figure is a tree with three top-level attributes:

```
figure
├── data      ← list of traces (the actual chart elements)
├── layout    ← title, axes, colours, fonts, legend, etc.
└── frames    ← animation frames (not covered in this course)
```

**Useful references:**
- [Plotly Python Figure Reference](https://plotly.com/python/reference/) — exhaustive attribute reference
- [Plotly Express API](https://plotly.com/python-api-reference/plotly.express.html) — high-level API docs
- [update_layout() reference](https://plotly.com/python/reference/layout/) — full layout customisation

> 📌 You do not need to populate every attribute. Plotly fills in sensible defaults for anything you leave unspecified.

## Two Ways to Create a Plotly Chart

| Approach | When to use | Code volume |
|---|---|---|
| **Plotly Express** (`px`) | Quick charts, standard plots, tidy data | Low |
| **Graph Objects** (`go`) | Full control, custom traces, multi-trace precision | Higher |

> ♟️ **Course strategy:** start with `px` for speed. Drop into `go` when you need control that `px` cannot provide.

In [ ]:
import pandas as pd

# Graph Objects — for full customisation
import plotly.graph_objects as go

# Plotly Express — for fast, clean charts
import plotly.express as px


---
## 2. Working with Data

We'll use the stocks dataset that ships with Plotly throughout this notebook.

In [ ]:
# Plotly ships with several built-in demo datasets
# px.data.<tab> to see them all: gapminder, iris, tips, stocks, etc.

df_stocks = px.data.stocks(datetimes=True)
df_stocks.head()


In [ ]:
df_stocks.info()


> 📌 **Wide vs Tidy data:** This dataset is in **wide format** (one column per company).
> Plotly Express works best with **tidy (long) format** — one row per observation.
> We'll convert it below when we need to plot all companies at once.

---
## 3. Creating a Figure with Graph Objects 🐌

Graph Objects give you maximum control. You build the figure piece by piece:
1. Create **traces** (the data series)
2. Define a **layout** (titles, axes, styling)
3. Combine into a **figure** with `go.Figure()`

In [ ]:
# Step 1 — create traces (data series)
# go.Scatter handles both line plots and scatter plots via the `mode` argument

trace1 = go.Scatter(
    x=df_stocks['date'],
    y=df_stocks['GOOG'],
    mode='lines',          # 'lines', 'markers', or 'lines+markers'
    name='Google'          # label in the legend
)

trace2 = go.Scatter(
    x=df_stocks['date'],
    y=df_stocks['MSFT'],
    mode='lines',
    name='Microsoft'
)

my_data = [trace1, trace2]


In [ ]:
# Step 2 — define layout (axes, title, etc.)
my_layout = {
    'title': 'Google vs Microsoft Stock Price',
    'xaxis': {'title': 'Date'},
    'yaxis': {'title': 'Closing Price (USD)'}
}


In [ ]:
# Step 3 — combine into a figure and show
figure = go.Figure(data=my_data, layout=my_layout)
figure.show()


In [ ]:
# You can update any part of the figure after creation
# update_layout() also re-renders the figure automatically

figure.update_layout(
    title='Google vs Microsoft — Weekly Closing Prices',
    title_font=dict(size=18, color='#2C2C2C', family='Arial')
)


---
## 4. Creating a Figure with Plotly Express 🚀

Plotly Express wraps the most common chart types in a single function call.
Same chart as above — in one line:

In [ ]:
fig = px.line(
    data_frame=df_stocks,
    x='date',
    y='GOOG',
    title='Google Stock Closing Price'
)
fig.show()


In [ ]:
# Adding common customisations directly in px
fig = px.line(
    data_frame=df_stocks,
    x='date',
    y='GOOG',
    title='Google Stock over Time',
    labels={'date': 'Date', 'GOOG': 'Closing Price (USD)'},   # rename columns in figure
    width=1000,
    height=450
)
fig.show()


### Plotly Express — Key Arguments

| Argument | Purpose | Example |
|---|---|---|
| `data_frame` | Source dataframe | `df` |
| `x` / `y` | Axis columns | `"date"`, `"sales"` |
| `color` | Colour grouping | `"region"`, `"company"` |
| `symbol` | Marker shape grouping | `"gender"` |
| `size` | Marker size (numeric column) | `"population"` |
| `hover_name` | Main hover label | `"country"` |
| `hover_data` | Extra hover info | `["profit", "rank"]` |
| `text` | Text shown on marks | `"label_col"` |
| `facet_row` / `facet_col` | Create small multiples | `"year"`, `"gender"` |
| `labels` | Rename axes / legend | `{"sales": "Revenue (€)"}` |
| `title` | Figure title | `"Monthly Sales"` |
| `template` | Visual theme | `"plotly_white"`, `"simple_white"` |
| `color_discrete_sequence` | Custom discrete colours | `px.colors.qualitative.Set2` |
| `color_continuous_scale` | Continuous colour scale | `px.colors.sequential.Blues` |
| `range_x` / `range_y` | Axis limits | `[0, 100]` |
| `log_x` / `log_y` | Log scale | `True` |
| `orientation` | Bar direction | `"h"` (horizontal) |
| `barmode` | Bar layout | `"group"`, `"stack"` |
| `opacity` | Transparency | `0.7` |
| `width` / `height` | Figure size in pixels | `900`, `500` |

---
## 5. Tidy Data + Plotly Express

Plotly Express works best with **tidy (long) format** — one row per observation, one column per variable.

Our stocks data is currently **wide**. We melt it to plot all companies at once:

In [ ]:
df_stocks.head()


In [ ]:
# pd.melt converts wide → long (tidy) format
df_stocks_tidy = pd.melt(
    df_stocks,
    id_vars='date',          # columns to keep as-is
    var_name='company',      # new column for the old column names
    value_name='price'       # new column for the values
)

print(f"Wide shape:  {df_stocks.shape}")
print(f"Tidy shape:  {df_stocks_tidy.shape}")
df_stocks_tidy.head(8)


In [ ]:
# Now we can plot ALL companies with a single px call
# color='company' creates one line per company automatically

fig = px.line(
    data_frame=df_stocks_tidy,
    x='date',
    y='price',
    color='company',
    title='Weekly Closing Prices — All Companies',
    labels={'date': 'Date', 'price': 'Closing Price (USD)', 'company': 'Company'},
    width=1000,
    height=450
)
fig.show()


---
## 6. Colours in Plotly

Plotly has built-in colour palettes for both discrete (categorical) and continuous data.

In [ ]:
# Built-in discrete colour palettes (for categorical data)
fig = px.colors.qualitative.swatches()
fig.show()


In [ ]:
# Built-in continuous colour scales (for numeric data)
fig = px.colors.sequential.swatches_continuous()
fig.show()


In [ ]:
# Plotly's default colour list (what you get when you don't specify colour)
print(px.colors.DEFAULT_PLOTLY_COLORS)


In [ ]:
# Using a custom discrete palette
fig = px.line(
    data_frame=df_stocks_tidy,
    x='date',
    y='price',
    color='company',
    color_discrete_sequence=px.colors.qualitative.Set2,  # swap palette here
    title='Weekly Closing Prices',
    labels={'date': 'Date', 'price': 'Closing Price (USD)', 'company': 'Company'},
    width=1000,
    height=450
)
fig.show()


---
## 7. Customising After Creation

Both `px` and `go` figures support three powerful update methods for post-creation customization:

| Method | What it controls |
|---|---|
| `fig.update_layout()` | Title, axes, legend, background, fonts, margins |
| `fig.update_traces()` | Line/marker styling, opacity, hover behaviour |
| `fig.update_xaxes()` / `fig.update_yaxes()` | Individual axis properties |

These work on **any** Plotly figure regardless of how it was created.

### `update_layout()` — Cheatsheet

| Argument | Controls | Example |
|---|---|---|
| `title` | Chart title text | `fig.update_layout(title="…")` |
| `title_x` | Title horizontal position (0–1) | `fig.update_layout(title_x=0.5)` |
| `title_font` | Title font styling | see example below |
| `xaxis_title` / `yaxis_title` | Axis labels | `fig.update_layout(xaxis_title="…")` |
| `xaxis_range` / `yaxis_range` | Axis limits | `fig.update_layout(xaxis_range=[0,100])` |
| `width` / `height` | Figure size in pixels | `fig.update_layout(width=900)` |
| `margin` | Padding (l, r, t, b in px) | `fig.update_layout(margin=dict(l=50,r=20,t=60,b=40))` |
| `legend` | Full legend config | see example below |
| `template` | Visual theme | `fig.update_layout(template="plotly_white")` |
| `font` | Default font for all text | see example below |
| `plot_bgcolor` | Chart area background | `fig.update_layout(plot_bgcolor="white")` |
| `paper_bgcolor` | Figure background | `fig.update_layout(paper_bgcolor="white")` |
| `hovermode` | Hover behaviour | `"closest"`, `"x unified"` |
| `barmode` | Bar chart layout | `"group"`, `"stack"`, `"overlay"` |

In [ ]:
# update_layout examples

# (1) title font
fig.update_layout(
    title_font=dict(
        family="Arial, sans-serif",
        size=20,
        color="darkslategrey"  # color can be provided in hex, rgb format, name string
    )
)

# (2) legend positioning
fig.update_layout(
    legend=dict(
        title="Company",
        orientation="h",    # horizontal legend
        x=0.0,
        y=1.1               # place above the chart
    )
)

# (3) global font
fig.update_layout(
    font=dict(
        family="Arial, sans-serif",
        size=12,
        color="darkgrey"
    )
)

# (4) clean white background (SWD standard)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)



### `update_traces()` — Cheatsheet

| Argument | Controls | Example |
|---|---|---|
| `mode` | Display mode | `"markers"`, `"lines"`, `"lines+markers"` |
| `marker_size` | Marker size | `fig.update_traces(marker_size=8)` |
| `marker_color` | Marker colour | `fig.update_traces(marker_color="#2E75B6")` |
| `marker_symbol` | Marker shape | `"circle"`, `"diamond"`, `"square"` |
| `line_width` | Line thickness | `fig.update_traces(line_width=2)` |
| `line_dash` | Line dash style | `"solid"`, `"dash"`, `"dot"`, `"dashdot"` |
| `opacity` | Trace transparency (0–1) | `fig.update_traces(opacity=0.7)` |
| `showlegend` | Show in legend | `fig.update_traces(showlegend=False)` |
| `marker_line_width` | Outline around markers | `fig.update_traces(marker_line_width=0)` |

In [ ]:
# update_traces examples

# (1) add markers to a line chart
fig.update_traces(
    mode="lines+markers",
    line_width=2,
    marker=dict(
        size=6,
        symbol="circle"
    )
)

# (2) line styling
fig.update_traces(
    line=dict(
        width=2.5,
        dash="solid"   # 'solid', 'dash', 'dot', 'dashdot'
    )
)



### `update_xaxes()` / `update_yaxes()` — Cheatsheet

| Attribute | Purpose | Example |
|---|---|---|
| `title_text` | Axis title | `"Sales (€)"` |
| `showgrid` | Show gridlines | `True`, `False` |
| `gridcolor` | Gridline colour | `"#EEEEEE"` |
| `zeroline` | Show zero line | `True`, `False` |
| `showline` | Show axis spine | `True`, `False` |
| `tickangle` | Rotate tick labels | `0`, `45`, `-90` |
| `tickformat` | Label format | `".2f"`, `"%Y"`, `",.0f"` |
| `range` | Axis range | `[0, 100]` |
| `type` | Scale type | `"linear"`, `"log"`, `"date"`, `"category"` |
| `visible` | Show/hide axis | `True`, `False` |

In [ ]:
# update_xaxes / update_yaxes examples

fig.update_xaxes(
    title_text="Date",
    showgrid=False,          # no vertical gridlines
    tickangle=0
)

fig.update_yaxes(
    title_text="Closing Price (USD)",
    showgrid=True,
    gridcolor="#EEEEEE",     # very light gridlines
    zeroline=False
)

fig.show()


---
## 8. Plotly Templates (Themes)

Templates control the default appearance of all chart elements — colours, fonts, gridlines, backgrounds.

> 💡 **Recommended template for this course:** `"plotly_white"` or `"simple_white"` — both align with the white-background, minimal-clutter standard from *Storytelling with Data*.

| Template | Description | Best for |
|---|---|---|
| `plotly` | Default: bright colours, light background | Quick exploration |
| `plotly_white` | Clean white, subtle gridlines | Dashboards, reports |
| `plotly_dark` | Dark background, high contrast | Dark-mode dashboards |
| `presentation` | Large fonts, minimal gridlines | Slides |
| `simple_white` | Very minimal, no gridlines | Academic / publications |
| `ggplot2` | R ggplot2-inspired | Users familiar with R |
| `seaborn` | Seaborn-inspired | Statistical / EDA |
| `gridon` | Strong gridlines | Precise value reading |

In [ ]:
# Applying a template globally to a figure
fig = px.line(
    data_frame=df_stocks_tidy,
    x='date', y='price', color='company',
    title='Stock Prices — simple_white theme',
    labels={'date': 'Date', 'price': 'Price (USD)', 'company': 'Company'},
    width=1000, height=450,
    template='simple_white'          # apply template here
)
fig.show()


In [ ]:
# You can also set a global default template for the entire session
import plotly.io as pio
pio.templates.default = "plotly_white"

# Now every figure uses plotly_white by default without specifying it each time
fig = px.line(df_stocks_tidy, x='date', y='price', color='company',
              title='This uses plotly_white by default')
fig.show()


---
## 9. Putting It All Together — A Complete Example

Here is the full workflow from raw data to a polished, professionally styled chart:

In [ ]:
# Full example: clean, professional line chart applying SWD + Plotly best practices

fig = px.line(
    data_frame=df_stocks_tidy,
    x='date',
    y='price',
    color='company',
    labels={'date': '', 'price': 'Closing Price (USD)', 'company': 'Company'},
    color_discrete_sequence=px.colors.qualitative.Set2
)

# Layout: white background, clean fonts, axis labels (SWD standard)
fig.update_layout(
    title='Tech stock prices have diverged sharply since 2019',
    title_font=dict(family='Arial', size=16, color='#2C2C2C'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color='#333333'),
    legend=dict(title='Company', orientation='h', y=1.08, x=0),
    margin=dict(l=60, r=40, t=70, b=40),
    width=1000,
    height=450
)

# Axes: minimal gridlines, no spines (SWD: remove clutter)
fig.update_xaxes(showgrid=False, showline=False)
fig.update_yaxes(gridcolor='#EEEEEE', showgrid=True, zeroline=False)

# Traces: consistent line width
fig.update_traces(line_width=2)

fig.show()


---
## Quick Reference Summary

```python
import plotly.express as px
import plotly.graph_objects as go

# ── Plotly Express (fast) ────────────────────────────────────────────────────
fig = px.line(df, x='date', y='value', color='group',
              title='...', labels={'value': 'Revenue'},
              template='plotly_white', width=1000, height=450)

# ── Post-creation customisation (works on any figure) ────────────────────────
fig.update_layout(
    title_font=dict(family='Arial', size=16),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(orientation='h', y=1.08)
)
fig.update_traces(line_width=2)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#EEEEEE')

# ── Graph Objects (full control) ──────────────────────────────────────────────
trace = go.Scatter(x=df['date'], y=df['value'], mode='lines', name='Series 1')
fig = go.Figure(data=[trace], layout={'title': '...', 'xaxis': {'title': 'Date'}})

fig.show()
```

> 📌 **Next lecture:** We start building real charts with real data — bar charts on the World Happiness Report. All the tools from this notebook will be our foundation.